In [ ]:
!git clone https://github.com/t-ravnushkin/generalized-toric-gpu-search.git

# GF(8) Toric Code Champion Search — Kaggle (NVIDIA GPU)

The OpenCL kernel is identical to the local Apple Silicon version — OpenCL runs natively on NVIDIA GPUs via the CUDA toolkit's OpenCL ICD.  
Upload `gf8.py`, `precompute.py`, `kernel.py`, and `champion_search.py` as a Kaggle Dataset, add it to this notebook, then run.

In [ ]:
# Install PyOpenCL (NVIDIA OpenCL ICD is already present in Kaggle's CUDA image)
!pip install pyopencl -q

# Add the uploaded dataset to sys.path so we can import the project modules
import sys, os
# Adjust the path below to wherever Kaggle mounts your dataset
DATASET_PATH = "/kaggle/input/gf8-toric-search"   # <-- change to your dataset slug
sys.path.insert(0, DATASET_PATH)

os.environ.setdefault("PYOPENCL_CTX", "0")
os.chdir("/kaggle/working")

In [ ]:
# Verify the GPU is visible to OpenCL
import pyopencl as cl
for p in cl.get_platforms():
    for d in p.get_devices():
        print(f"{p.name}  |  {d.name}  |  type={'GPU' if d.type == cl.device_type.GPU else 'CPU'}")

In [ ]:
from precompute import init_opencl, bounding_cube_points
from kernel import DistanceOracle
from champion_search import (
    find_latest_results,
    load_results,
    check_set,
    global_batched_bfs,
    idx,
)
from pathlib import Path
from datetime import datetime

ctx, queue_cl, M_buf, _ = init_opencl()
oracle = DistanceOracle(ctx, queue_cl, M_buf)
lattice = bounding_cube_points()

In [ ]:
# Pick up the most recent results file in /kaggle/working, or start fresh
results_file = find_latest_results()
if results_file is None:
    results_file = Path(f"champions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    print(f"Starting fresh: {results_file}")
else:
    existing = load_results(results_file)
    print(f"Resuming from: {results_file}  ({len(existing)} records)")

## Part A — Structured geometric candidates

In [ ]:
already_done = (
    {rec["name"] for rec in load_results(results_file) if "name" in rec}
    if results_file.exists() else set()
)

circle_8 = sorted([
    idx(0,1), idx(0,6), idx(1,0), idx(2,2),
    idx(2,5), idx(5,2), idx(5,5), idx(6,0),
])
check_set("circle_conic_k8", circle_8, oracle, lattice, results_file, already_done=already_done)

parabola_7 = sorted([idx(t, (t * t) % 7) for t in range(7)])
check_set("parabola_conic_k7", parabola_7, oracle, lattice, results_file, already_done=already_done)

## Part B — Global BFS (resumes automatically)

In [ ]:
import time
t0 = time.perf_counter()
global_batched_bfs(
    target_distance=30,
    oracle=oracle,
    lattice=lattice,
    results_file=results_file,
    max_k=8,
    resume=True,
)
print(f"\nTotal BFS time: {time.perf_counter() - t0:.1f}s")

## Inspect results

In [ ]:
import pandas as pd

records = [r for r in load_results(results_file) if r.get("type") != "level_complete"]
df = pd.DataFrame(records)[["name", "k", "min_distance", "max_zeros", "indices"]]
df.sort_values(["k", "min_distance"], ascending=[True, False], inplace=True)
df.reset_index(drop=True, inplace=True)
display(df)